# Phase 2 — Shrinking the Model for Constrained Deployment

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, recall_score, precision_score, f1_score

df = pd.read_csv('ai4i2020.csv')

data = df.drop(columns=['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'])
le = LabelEncoder()
data['Type'] = le.fit_transform(data['Type'])

X = data.drop(columns=['Machine failure'])
y = data['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## Baseline (from Phase 1, retrained here for comparison)
200 trees, no depth limit, class_weight='balanced'

In [2]:
baseline = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)
baseline.fit(X_train, y_train)

joblib.dump(baseline, 'model_baseline.pkl')
baseline_size = os.path.getsize('model_baseline.pkl') / 1024

y_pred = baseline.predict(X_test)
baseline_metrics = {
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'size_kb': baseline_size
}
baseline_metrics

{'precision': 0.8780487804878049,
 'recall': 0.5294117647058824,
 'f1': 0.6605504587155964,
 'size_kb': 6330.0712890625}

## Variant A — Fewer trees + limited depth
50 trees, max_depth=8

In [3]:
variant_a = RandomForestClassifier(
    n_estimators=50,
    max_depth=8,
    class_weight='balanced_subsample',
    random_state=42
)
variant_a.fit(X_train, y_train)

joblib.dump(variant_a, 'model_variant_a.pkl')
size_a = os.path.getsize('model_variant_a.pkl') / 1024

y_pred_a = variant_a.predict(X_test)
metrics_a = {
    'precision': precision_score(y_test, y_pred_a),
    'recall': recall_score(y_test, y_pred_a),
    'f1': f1_score(y_test, y_pred_a),
    'size_kb': size_a
}
metrics_a

{'precision': 0.5473684210526316,
 'recall': 0.7647058823529411,
 'f1': 0.6380368098159509,
 'size_kb': 696.6337890625}

## Variant B — Even smaller + compressed file
20 trees, max_depth=6, joblib compression

In [4]:
variant_b = RandomForestClassifier(
    n_estimators=20,
    max_depth=6,
    class_weight='balanced_subsample',
    random_state=42
)
variant_b.fit(X_train, y_train)

joblib.dump(variant_b, 'model_variant_b.pkl', compress=3)
size_b = os.path.getsize('model_variant_b.pkl') / 1024

y_pred_b = variant_b.predict(X_test)
metrics_b = {
    'precision': precision_score(y_test, y_pred_b),
    'recall': recall_score(y_test, y_pred_b),
    'f1': f1_score(y_test, y_pred_b),
    'size_kb': size_b
}
metrics_b

{'precision': 0.3783783783783784,
 'recall': 0.8235294117647058,
 'f1': 0.5185185185185185,
 'size_kb': 55.8408203125}

## Variant C — Fix recall with a lower decision threshold
Same small model as Variant B, but instead of the default 0.5 cutoff,
lower the threshold so it flags "failure" more readily (trades some precision for recall —
usually the right trade for fault detection, since missing a real failure is worse than a false alarm)

In [5]:
probs = variant_b.predict_proba(X_test)[:, 1]
threshold = 0.3
y_pred_c = (probs >= threshold).astype(int)

metrics_c = {
    'precision': precision_score(y_test, y_pred_c),
    'recall': recall_score(y_test, y_pred_c),
    'f1': f1_score(y_test, y_pred_c),
    'size_kb': size_b
}
metrics_c

{'precision': 0.25,
 'recall': 0.9264705882352942,
 'f1': 0.39375,
 'size_kb': 55.8408203125}

## Compare all variants

In [6]:
comparison = pd.DataFrame({
    'baseline': baseline_metrics,
    'variant_a (50 trees, depth 8)': metrics_a,
    'variant_b (20 trees, depth 6, compressed)': metrics_b,
    'variant_c (variant_b + threshold 0.3)': metrics_c,
}).T

comparison

,precision,recall,f1,size_kb
baseline,0.878049,0.529412,0.660550,6330.071289
"variant_a (50 trees, depth 8)",0.547368,0.764706,0.638037,696.633789
"variant_b (20 trees, depth 6, compressed)",0.378378,0.823529,0.518519,55.840820
variant_c (variant_b + threshold 0.3),0.250000,0.926471,0.393750,55.840820


## Pick the winner and save as the final constrained model
Variant C: smallest file size (from variant_b's compressed weights) with recall
recovered via threshold tuning — this is what moves forward into Docker/FastAPI.

In [7]:
joblib.dump(variant_b, 'fault_detector_constrained.pkl', compress=3)
final_threshold = threshold

with open('model_config.txt', 'w') as f:
    f.write(f'threshold={final_threshold}\n')

print('Saved fault_detector_constrained.pkl, size:', os.path.getsize('fault_detector_constrained.pkl') / 1024, 'KB')
print('Decision threshold to use at inference:', final_threshold)

Saved fault_detector_constrained.pkl, size: 55.8408203125 KB
Decision threshold to use at inference: 0.3


## Takeaways
- Baseline (200 trees) vs constrained model (20 trees, depth 6, compressed): compare size_kb in the table above
- Lowering the decision threshold to 0.3 recovers recall lost from shrinking the model
- `fault_detector_constrained.pkl` + `model_config.txt` (threshold) are what get loaded into the FastAPI service in Phase 3